# 02 — Feature Engineering

Build the **27 account-level features used by the final cross-validation pipeline**.

Experimental feature blocks from the competition-development notebook were removed
from this public version to keep the final implementation focused and readable.

In [ ]:
from functools import reduce
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARTIFACT_DIR = REPO_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

df_base = pd.read_parquet(ARTIFACT_DIR / "df_feature.parquet")
features_list = []

In [ ]:
# 1. Basic transaction statistics
basic_stats = (
    df_base.groupby("acct")
    .agg(
        sum_txn_amt_twd=("txn_amt_twd", "sum"),
        total_txn_cnt=("acct", "count"),
    )
    .reset_index()
)
features_list.append(basic_stats)

In [ ]:
# 2. Inbound/outbound transaction ratios
direction_ratio = (
    df_base.groupby("acct")["direction"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
)
direction_ratio.columns = [f"dir_{col}" for col in direction_ratio.columns]
features_list.append(direction_ratio.reset_index())

# 3. Self-transaction ratios
self_txn_ratio = (
    df_base.groupby("acct")["is_self_txn"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
)
self_txn_ratio.columns = [f"is_self_txn_{col}" for col in self_txn_ratio.columns]
features_list.append(self_txn_ratio.reset_index())

In [ ]:
# 4. Temporal behavior
df_base["is_deep_night"] = ((df_base["hour"] >= 0) & (df_base["hour"] < 6)).astype(int)
df_base["is_business_hour"] = ((df_base["hour"] >= 9) & (df_base["hour"] <= 18)).astype(int)

time_features = (
    df_base.groupby("acct")
    .agg(
        deep_night_ratio=("is_deep_night", "mean"),
        business_hour_ratio=("is_business_hour", "mean"),
        txn_date_min=("txn_date", "min"),
        txn_date_max=("txn_date", "max"),
        txn_count=("txn_date", "size"),
        unique_days=("txn_date", "nunique"),
    )
    .reset_index()
)

time_features["txn_day_span"] = (
    time_features["txn_date_max"] - time_features["txn_date_min"] + 1
)
time_features["avg_txn_per_day"] = (
    time_features["txn_count"] / time_features["unique_days"]
)

hour_concentration = (
    df_base.groupby("acct")["hour"]
    .apply(lambda x: x.value_counts().max() / len(x))
    .reset_index(name="hour_concentration")
)

time_features = (
    time_features.merge(hour_concentration, on="acct", how="left")
    .drop(columns=["txn_date_min", "txn_date_max", "txn_count"])
)
features_list.append(time_features)

In [ ]:
# 5. Amount statistics by transaction direction
amt_by_dir = (
    df_base.groupby(["acct", "direction"])["txn_amt_twd"]
    .agg(["sum", "mean", "std"])
    .unstack(fill_value=0)
)
amt_by_dir.columns = [
    "amt_twd_" + "_".join(col).strip()
    for col in amt_by_dir.columns.values
]
std_cols = [col for col in amt_by_dir.columns if "std" in col]
amt_by_dir[std_cols] = amt_by_dir[std_cols].fillna(0)
features_list.append(amt_by_dir.reset_index())

# 6. Net fund flow
flow = (
    df_base.groupby(["acct", "direction"])["txn_amt_twd"]
    .sum()
    .unstack(fill_value=0)
)
net_flow = flow.get("in", 0) - flow.get("out", 0)
features_list.append(net_flow.rename("net_flow_twd").reset_index())

In [ ]:
# 7. Counterparty-network behavior
network_in = (
    df_base[df_base["direction"] == "in"]
    .groupby("acct")["counterparty"].nunique()
    .reset_index(name="deg_in")
)
network_out = (
    df_base[df_base["direction"] == "out"]
    .groupby("acct")["counterparty"].nunique()
    .reset_index(name="deg_out")
)
network_all = (
    df_base.groupby("acct")["counterparty"].nunique()
    .reset_index(name="deg_all")
)

network_conc = (
    df_base.groupby(["acct", "counterparty"])["txn_amt_twd"]
    .sum()
    .reset_index()
    .sort_values(["acct", "txn_amt_twd"], ascending=[True, False])
)

def calc_top3_concentration(group: pd.DataFrame) -> float:
    total = group["txn_amt_twd"].sum()
    if total <= 0:
        return 0.0
    return float(group.head(3)["txn_amt_twd"].sum() / total)

network_conc = (
    network_conc.groupby("acct", group_keys=False)
    .apply(calc_top3_concentration, include_groups=False)
    .reset_index(name="top3_counterparty_concentration")
)

features_list.extend([network_in, network_out, network_all, network_conc])

In [ ]:
# 8. Transaction velocity
unique_days = df_base.groupby("acct")["txn_date"].nunique()

velocity = (
    df_base.groupby("acct")
    .agg(
        total_txn_count=("acct", "count"),
        total_amt_twd=("txn_amt_twd", "sum"),
    )
    .reset_index()
)
velocity["unique_days"] = velocity["acct"].map(unique_days).fillna(1)
velocity["txn_per_day"] = velocity["total_txn_count"] / velocity["unique_days"]
velocity["amt_per_day_twd"] = velocity["total_amt_twd"] / velocity["unique_days"]
features_list.append(velocity[["acct", "txn_per_day", "amt_per_day_twd"]])

# 9. Account-type ratios
acct_type_ratio = (
    df_base.groupby("acct")["acct_type"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
)
acct_type_ratio.columns = [f"acct_type_{col}" for col in acct_type_ratio.columns]
features_list.append(acct_type_ratio.reset_index())

In [ ]:
# Merge all feature blocks.
df_features = reduce(
    lambda left, right: pd.merge(left, right, on="acct", how="outer"),
    features_list,
)
df_features = (
    df_features
    .fillna(0)
    .replace([np.inf, -np.inf], 0)
)

feature_cols = [col for col in df_features.columns if col != "acct"]
print(f"Feature count: {len(feature_cols)}")
print(feature_cols)

EXPECTED_FEATURE_COUNT = 27
if len(feature_cols) != EXPECTED_FEATURE_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_FEATURE_COUNT} features from the competition schema, "
        f"but found {len(feature_cols)}. Check categorical values in the input data."
    )

In [ ]:
output_path = ARTIFACT_DIR / "df_feature_final.parquet"
df_features.to_parquet(output_path, index=False)
print(f"Saved: {output_path}")